<a href="https://colab.research.google.com/github/heberdavi/mba-engsoft-tcc/blob/main/04_inventario_processamento.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Célula 1: Preparação do ambiente
# Instala o Streamlit e o Localtunnel (para gerar o link público)
!pip install -q streamlit
!npm install -q -g localtunnel

# Monta o Google Drive
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.6 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸Mounted at /content/drive


In [2]:
%%writefile app_inventario.py
import streamlit as st
import pandas as pd
import sqlite3
import plotly.express as px

# Configuração da página para aproveitar toda a largura da tela
st.set_page_config(layout="wide", page_title="Inventário de Auditoria - MBA")

# Função com Cache para performance do pipeline
@st.cache_data
def load_data():
    path = "/content/drive/MyDrive/mba-engsof-tcc/versao_final/data/base-dados.db"
    conn = sqlite3.connect(path)

    query = """
      with corpus as (
      select
        gl.id genero_id, gl.nome genero,
        l.nome nome_livro, l.abreviacao abreviacao_livro,
        v.numero_capitulo capitulo, v.numero_verso versiculo, v.texto texto,
        t.antidoto_referencia topico_classificacao_final,
        vt.similaridade_final topico_score_final,
        case vs.sentimento_num when 0 then 'Neutro' when 1 then 'Positivo' when -1 then 'Negativo' end sentimento_classificacao_final,
        max(vs.score_pos, vs.score_neg, vs.score_neu) sentimento_score_final,
        -- IA EXPLICAVEL --
        vl.texto_limpo, length(v.texto) tamanho_texto,
        case max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio)
          when vt.p_exaustao then 'Exaustão vs. Refrigério'
          when vt.p_transitoriedade then 'Transitoriedade vs. Solidez'
          when vt.p_vazio then 'Vazio vs. Propósito'
        end topico_melhor_classificacao_existencial,
        max(vt.p_exaustao, vt.p_transitoriedade, vt.p_vazio) topico_melhor_score_existencial,
        vt.p_exaustao topico_score_exaustao, vt.p_transitoriedade topico_score_transitoriedade, vt.p_vazio topico_score_vazio, vt.p_narrativo topico_score_narrativo,
        vs.score_pos sentimento_score_positivo, vs.score_neg sentimento_score_negativo, vs.score_neu sentimento_score_neutro,
        -- calculo da margem de dominancia (melhor score dos eixos existenciais, menos o score do eixo narrativo)
        vt.margem_dominancia,
        -- grau de incerteza da decisao
        -- quanto mais baixo, maior a certeza do enquadramento no eixo escolhido
        -- quanto mais alto, maior a dúvida entre os eixos
        vt.entropia,
        -- distância entre as duas maiores probabilidades
        vt.gap_confianca,
        -- decisão final
        vt.status_decisao decisao_final
      from genero_literario gl
      join livro l on l.genero_id = gl.id
      join verso v on v.livro_id = l.id
      join verso_limpo vl on vl.verso_id = v.id
      join verso_sentimento vs on vs.verso_id = v.id
      join verso_topico vt on vt.verso_id = v.id
      join topico t on t.id = vt.topico_id)
      select genero, nome_livro, abreviacao_livro,
        capitulo, versiculo, texto,
        topico_classificacao_final, topico_score_final,
        sentimento_classificacao_final, sentimento_score_final,
        -- auditoria do texto
        tamanho_texto, texto_limpo,
        -- auditoria da modelagem de tópicos
        topico_melhor_classificacao_existencial,
        topico_melhor_score_existencial,
        topico_score_exaustao, topico_score_transitoriedade, topico_score_vazio, topico_score_narrativo,
        margem_dominancia, entropia, gap_confianca,
        -- auditoria da análise de sentimentos
        sentimento_score_positivo, sentimento_score_negativo, sentimento_score_neutro,
        -- decisao
        decisao_final,
        case
          when tamanho_texto < 35 and margem_dominancia < 0.25
              then 'Texto muito curto'
          else
              case
              when genero_id in (1, 2) and topico_melhor_score_existencial > 0.88 and margem_dominancia > 0.15
            then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.88). A margem de dominância ('||round(margem_dominancia, 2)||') também superou a referência para o eixo (0.15)'
              when genero_id in (3, 4) and topico_melhor_score_existencial > 0.50
                  then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.50)'
              when genero_id in (5, 6) and (topico_melhor_score_existencial > 0.60 or (topico_melhor_score_existencial > 0.45 and margem_dominancia > 0.10)) then
                  case
                      when topico_melhor_score_existencial > 0.60
                          then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.60)'
                      else 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.45). A margem de dominância ('||round(margem_dominancia, 2)||') também superou a referência para o eixo (0.10)'
                  end
              when genero_id = 7 and topico_melhor_score_existencial > 0.75
                  then 'Classificação existencial ('||round(topico_melhor_score_existencial, 2)||') no eixo '||topico_melhor_classificacao_existencial||' superou o score de referência para o gênero (0.75)'
              else
                  'Regra geral'
              end
          end motivo_decisao
      from corpus
    """
    df = pd.read_sql(query, conn)
    conn.close()
    return df

df = load_data()

# --- INTERFACE STREAMLIT ---
st.title("🛡️ Inventário de Auditoria: Classificação de Antídotos")
st.markdown("Interface técnica para validação de regras de negócio e explicabilidade do modelo (XAI).")

# Sidebar para filtros
with st.sidebar:
    st.header("Painel de Controle")
    # Busca por palavra-chave no texto original
    busca = st.text_input("Buscar termo no verso (ex: vida, alma)")
    # Filtro de Confiança (Slider)
    confianca_min = st.slider("Confiança Mínima (Score)", 0.0, 1.0, 0.0)
    # Filtro de Sentimento
    sentimento_opcoes = st.multiselect("Filtrar Sentimento", options=sorted(df['sentimento_classificacao_final'].unique()))
    # Filtros de Gênero e Motivo
    genero_list = st.multiselect("Filtrar Gênero", options=sorted(df['genero'].unique()))
    status_list = st.multiselect("Decisão Final", options=sorted(df['decisao_final'].unique()))

# Aplicação dos Filtros
df_view = df.copy()
if busca:
    df_view = df_view[df_view['texto'].str.contains(busca, case=False, na=False)]
if confianca_min > 0:
    df_view = df_view[df_view['topico_melhor_classificacao_existencial'] >= confianca_min]
if sentimento_opcoes:
    df_view = df_view[df_view['sentimento_classificacao_final'].isin(sentimento_opcoes)]
if genero_list:
    df_view = df_view[df_view['genero'].isin(genero_list)]
if status_list:
    df_view = df_view[df_view['decisao_final'].isin(status_list)]

# Tabela de Dados Interativa
st.subheader(f"Registros Encontrados: {len(df_view)}")
st.dataframe(df_view, use_container_width=True)

# Inspeção Detalhada com Validação de Dados
st.divider()
st.subheader("🔍 Inspeção de IA Explicável (XAI)")

if not df_view.empty:
    selected_verso = st.selectbox(
        "Escolha um verso para detalhamento:",
        options=df_view['texto'].tolist(),
        key="auditoria_selector"
    )

    # Busca o item garantindo que ele exista no frame filtrado
    item_match = df_view[df_view['texto'] == selected_verso]

    if not item_match.empty:
        item = item_match.iloc[0]
        c1, c2 = st.columns([1, 1])

        with c1:
            st.markdown(f"**📖 Gênero:** {item['genero']} | **Texto:** {item['abreviacao_livro']} {item['capitulo']}:{item['versiculo']}")
            st.info(f"**Texto:** {item['texto']}")

            # Formatação visual da justificativa
            if "Regra geral" in str(item['motivo_decisao']) or "muito curto" in str(item['motivo_decisao']):
                st.error(f"**Justificativa:** {item['motivo_decisao']}")
            else:
                st.success(f"**Justificativa:** {item['motivo_decisao']}")

        with c2:
            # Preparação e exibição do gráfico de barras horizontais
            chart_data = pd.DataFrame({
                'Eixo Existencial': ['Exaustão vs. Refrigério', 'Transitoriedade vs. Solidez', 'Vazio vs. Propósito', 'Narrativo/Normativo'],
                'Score (0-1)': [
                    float(item['topico_score_exaustao']),
                    float(item['topico_score_transitoriedade']),
                    float(item['topico_score_vazio']),
                    float(item['topico_score_narrativo'])
                ]
            })

            fig = px.bar(
                chart_data,
                x='Score (0-1)',
                y='Eixo Existencial',
                orientation='h',
                title="Probabilidades por Eixo",
                range_x=[0,1],
                color='Score (0-1)',
                color_continuous_scale='Blues'
            )
            fig.update_layout(showlegend=False, height=350)
            st.plotly_chart(fig, use_container_width=True)
    else:
        st.info("🔄 Atualizando visualização...")
else:
    st.warning("⚠️ Nenhum registro encontrado para os filtros selecionados na barra lateral.")

Writing app_inventario.py


In [3]:
# Célula 3: Execução e túnel de acesso com limpeza de sessão
!pip install -q pyngrok

import os
from pyngrok import ngrok
from google.colab import userdata

# 1. Limpeza de processos antigos para evitar o erro de limite de endpoints (ERR_NGROK_324)
ngrok.kill()
!pkill ngrok

# 2. Configuração do Token a partir do Secrets do Colab
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

# 3. Execução do Streamlit em background
# Redirecionamos a saída para um log para não poluir o notebook
!nohup streamlit run app_inventario.py > streamlit.log 2>&1 &

# 4. Abertura do Túnel com tratamento de exceção
try:
    # Definir um nome para o túnel ajuda a manter a sessão organizada
    public_url = ngrok.connect(8501, name="inventario_auditoria")
    print(f"✅ Sucesso! Inventário online.")
    print(f"🔗 Clique aqui para abrir: {public_url}")
except Exception as e:
    print(f"❌ Erro ao conectar o túnel: {e}")
    print("\n💡 DICA: Se o erro de limite persistir, acesse https://dashboard.ngrok.com/tunnels/agents e encerre as sessões ativas manualmente.")

✅ Sucesso! Inventário online.
🔗 Clique aqui para abrir: NgrokTunnel: "https://dimmer-boozy-gainfully.ngrok-free.dev" -> "http://localhost:8501"
